In [ ]:
import pandas as pd

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Yes, you can substitute almond milk for regular milk in pancakes. Almond milk is a great dairy-free alternative and will work well in most pancake recipes. Just use the same amount of almond milk as the recipe calls for regular milk. Keep in mind that almond milk has a slightly different flavor and consistency, so it may slightly alter the taste of your pancakes, but they should still turn out delicious! If you have any specific pancake recipe in mind, I could provide more tailored advice.


In [5]:
df_results_judged = pd.read_json('results_judged_20260617_205710.json')
print(df_results_judged.human_label.value_counts())
print(df_results_judged.judge_label.value_counts())

human_label
good    14
bad      6
Name: count, dtype: int64
judge_label
good    19
bad      1
Name: count, dtype: int64


In [6]:
def calculate_metrics(df):
    # True positive (TP): judge says "bad" AND human says "bad"
    tp = ((df.judge_label == 'bad') & (df.human_label == 'bad')).sum()
    # False positive (FP): judge says "bad" BUT human says "good"
    fp = ((df.judge_label == 'bad') & (df.human_label == 'good')).sum()
    # False negative (FN): judge says "good" BUT human says "bad"
    fn = ((df.judge_label == 'good') & (df.human_label == 'bad')).sum()
    # True negative (TN): judge says "good" AND human says "good"
    tn = ((df.judge_label == 'good') & (df.human_label == 'good')).sum()
    total = len(df)
    
    accuracy = (tp + tn) / total
    precision = tp / (tp + fp)  # when judge says "bad", how often is it right?
    recall = tp / (tp + fn)     # of all actual "bad", how many did the judge catch?
    
    print('accuracy:', accuracy)
    print('precision:', precision)
    print('recall', recall)

In [7]:
calculate_metrics(df_results_judged)

accuracy: 0.65
precision: 0.0
recall 0.0


In [11]:
def print_bad_label(df_results_judged):
    mask_false_predictions = (
        (
            (df_results_judged.judge_label == "bad") &
            (df_results_judged.human_label == "good")
        )
        |
        (
            (df_results_judged.judge_label == "good") &
            (df_results_judged.human_label == "bad")
        )
        |
        (
            (df_results_judged.judge_label == "bad") &
            (df_results_judged.human_label == "bad")
        )
    )
    
    for question, output, human_label, human_comment, judge_label, reasoning in zip(
        df_results_judged[mask_false_predictions].question,
        df_results_judged[mask_false_predictions].output,
        df_results_judged[mask_false_predictions].human_label,
        df_results_judged[mask_false_predictions].human_comments,
        df_results_judged[mask_false_predictions].judge_label,
        df_results_judged[mask_false_predictions].judge_reasoning,
    ):
        print(f"Question: {question}")
        print(f"Human label:{human_label}, judge label:{judge_label}")
        print(f"Human comment:\n{human_comment}")
        print(f"Judge reasoning:\n{reasoning}")
        print(f"\nOutput: {output}")
        print("-" * 80)

In [12]:
print_bad_label(df_results_judged)

Question: Please evaluate my answer for video_id wjZofJX0v4M. Tutor question: How are tokens turned into something the transformer can work with? My answer: The tokens are converted into numbers or vectors called embeddings, which place them in a high-dimensional space so the network can do mathematical operations on them.
Human label:bad, judge label:good
Human comment:
Would be nice if the agent asks follow-up questions on missing concepts instead of highlighting that it is missing.
Judge reasoning:
The agent adequately evaluated the learner's answer by confirming that it correctly identified embeddings as numerical vectors, fulfilling the criteria. The feedback provided is useful, highlighting specific areas for improvement such as the omission of the tokenization process and the need for a deeper understanding of how embeddings encode meaning. The suggestions are focused and relevant. The tool behavior was appropriate, utilizing evaluate_user_answer as required and using search_vid